In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path


In [38]:
def get_max_lambda(metric, computed_path):
    files = sorted(Path(computed_path + metric).glob("*.json"))
    max_lamda = []

    for f in files:
        df = pd.read_json(f)
        if(metric =="mexa"):
            df = df.T
        percentages = {i: df.loc[i].mean() * 100 for i in range(17)}
        
        best_layer = max(percentages, key=percentages.get)
        # print(percentages)
        if percentages[best_layer] == 0:
            max_lamda.append(None)
        else:
            max_lamda.append(best_layer)
    return max_lamda


In [39]:
dali_directory = Path('../alignment_results/Llama3.2-1B/belebele/dali')
tlang = [lang.name.split("_")[1] for lang in dali_directory.glob("*.json")]

belebele_path = "../alignment_results/Llama3.2-1B/belebele/"
dali_max_lamda = get_max_lambda("dali", belebele_path)
dalist_max_lamda = get_max_lambda("dali_strong", belebele_path)
mexa_max_lamda = get_max_lambda("mexa", belebele_path)

belebele_max_lamda_df = pd.DataFrame({
    "target_languages": tlang,
    "dali_max_lamda": dali_max_lamda,
    "dalist_max_lamda": dalist_max_lamda,
    "mexa_max_lamda": mexa_max_lamda
})
belebele_max_lamda_df

,target_languages,dali_max_lamda,dalist_max_lamda,mexa_max_lamda
0,deu,10,6,5
1,fra,10,5,5
2,hin,7,4,8
3,ita,7,7,5
4,por,10,5,5
5,spa,10,5,5


In [40]:
dali_directory = Path('../alignment_results/Llama3.2-1B/xcopa/dali')
tlang = [lang.name.split("_")[1] for lang in dali_directory.glob("*.json")]

xcopa_path = "../alignment_results/Llama3.2-1B/xcopa/"
dali_max_lamda = get_max_lambda("dali", xcopa_path)
dalist_max_lamda = get_max_lambda("dali_strong", xcopa_path)
mexa_max_lamda = get_max_lambda("mexa", xcopa_path)

xcopa_max_lamda_df = pd.DataFrame({
    "target_languages": tlang,
    "dali_max_lamda": dali_max_lamda,
    "dalist_max_lamda": dalist_max_lamda,
    "mexa_max_lamda": mexa_max_lamda
})
xcopa_max_lamda_df

,target_languages,dali_max_lamda,dalist_max_lamda,mexa_max_lamda
0,italian,5,0,7
1,thai,5,5,5


The dalist max lamda is 0  that's mean NA value as dalist is very strict so all of the 17 lamda is 0

In [ ]:
import pandas as pd

def get_xcopa_ts_tf(lang, num_samples=None):
    base = "../results/accuracy/Llama3.2-1B/xcopa"
    eng_df = pd.read_json(f"{base}/xcopa_eval_Llama3.2-1B_translation-{lang}_AB.jsonl", lines=True)
    lang_df = pd.read_json(f"{base}/xcopa_eval_Llama3.2-1B_{lang}_AB.jsonl", lines=True)
    print(lang_df)
    if num_samples is not None:
        eng_df = eng_df[eng_df["idx"] < num_samples]
        lang_df = lang_df[lang_df["idx"] < num_samples]

    merged = eng_df[["idx", "acc"]].merge(
        lang_df[["idx", "acc"]],
        on="idx",
        suffixes=("_eng", "_lang")
    )

    ts_idx = merged.loc[(merged["acc_eng"] == 1.0) & (merged["acc_lang"] == 1.0), "idx"].tolist()
    tf_idx = merged.loc[(merged["acc_eng"] == 1.0) & (merged["acc_lang"] == 0.0), "idx"].tolist()

    print(f"{lang.upper()} | Eng Acc: {eng_df['acc'].mean()*100:.1f}% | Lang Acc: {lang_df['acc'].mean()*100:.1f}% | TS: {len(ts_idx)} | TF: {len(tf_idx)}")
    return ts_idx, tf_idx

it_ts, it_tf = get_xcopa_ts_tf("it")
th_ts, th_tf = get_xcopa_ts_tf("th")

     idx  acc           selected_logits  correct_ans
0      0    0     [12.0234375, 12.0625]            0
1      1    1  [11.1953125, 10.9765625]            0
2      2    0  [11.6484375, 11.3359375]            1
3      3    1  [11.6328125, 11.1328125]            0
4      4    0    [11.59375, 11.7265625]            0
..   ...  ...                       ...          ...
495  495    1  [11.4765625, 11.5859375]            1
496  496    0    [11.4296875, 11.34375]            1
497  497    0     [11.359375, 11.78125]            0
498  498    0  [11.6171875, 11.3203125]            1
499  499    0     [11.671875, 11.65625]            1

[500 rows x 4 columns]
IT | Eng Acc: 56.8% | Lang Acc: 50.8% | TS: 155 | TF: 129
     idx  acc          selected_logits  correct_ans
0      0    1   [8.3515625, 8.0546875]            0
1      1    1   [8.1484375, 7.2421875]            0
2      2    0         [8.0, 7.5546875]            1
3      3    1  [7.3359375, 6.58984375]            0
4      4    1        [

In [15]:
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

def separate_ts_tf(lang, llm_name="Llama3.2-1B", base_dir="../results/accuracy"):
    folder = f"{base_dir}/{llm_name}/xcopa"
    eng_df = pd.read_json(f"{folder}/xcopa_eval_{llm_name}_translation-{lang}_AB.jsonl", lines=True)
    lang_df = pd.read_json(f"{folder}/xcopa_eval_{llm_name}_{lang}_AB.jsonl", lines=True)

    # Merge English translation and target language results on sample 'idx' (0..499)
    merged = eng_df[["idx", "acc"]].merge(
        lang_df[["idx", "acc"]],
        on="idx",
        suffixes=("_eng", "_lang")
    )

    # Separate into TS (correct in both) and TF (correct in English, wrong in target)
    ts_df = merged[(merged["acc_eng"] == 1.0) & (merged["acc_lang"] == 1.0)].reset_index(drop=True)
    tf_df = merged[(merged["acc_eng"] == 1.0) & (merged["acc_lang"] == 0.0)].reset_index(drop=True)

    return ts_df, tf_df

# Example for Italian ('it') and Thai ('th')
it_ts_df, it_tf_df = separate_ts_tf("it")
th_ts_df, th_tf_df = separate_ts_tf("th")

print(f"Italian -> TS: {len(it_ts_df)} rows | TF: {len(it_tf_df)} rows")
print(f"Thai    -> TS: {len(th_ts_df)} rows | TF: {len(th_tf_df)} rows")

Italian -> TS: 155 rows | TF: 129 rows
Thai    -> TS: 187 rows | TF: 72 rows


In [33]:
import pandas as pd
from pathlib import Path
from statsmodels.stats.proportion import proportions_ztest

def separate_ts_tf(lang_code, num_samples=167, llm_name="Llama3.2-1B", base_dir="../results/accuracy"):
    folder = f"{base_dir}/{llm_name}/xcopa"
    eng_df = pd.read_json(f"{folder}/xcopa_eval_{llm_name}_translation-{lang_code}_AB.jsonl", lines=True)
    lang_df = pd.read_json(f"{folder}/xcopa_eval_{llm_name}_{lang_code}_AB.jsonl", lines=True)

    # Restrict to the 167 samples present in your alignment JSONs
    eng_df = eng_df[eng_df["idx"] < num_samples]
    lang_df = lang_df[lang_df["idx"] < num_samples]

    merged = eng_df[["idx", "acc"]].merge(
        lang_df[["idx", "acc"]],
        on="idx",
        suffixes=("_eng", "_lang")
    )

    ts_df = merged[(merged["acc_eng"] == 1.0) & (merged["acc_lang"] == 1.0)].reset_index(drop=True)
    tf_df = merged[(merged["acc_eng"] == 1.0) & (merged["acc_lang"] == 0.0)].reset_index(drop=True)
    return ts_df, tf_df

def compute_delta_from_dfs(align_json_path, ts_df, tf_df, lambda_max):
    align_df = pd.read_json(align_json_path)

    # Ensure rows = layers (17) and columns = samples (167)
    if align_df.shape[0] != 17 and align_df.shape[1] == 17:
        align_df = align_df.T

    # Handle 0-based (0..166) vs 1-based (1..167) column keys automatically
    offset = 1 if 0 not in align_df.columns else 0
    ts_cols = [c for c in (ts_df["idx"] + offset) if c in align_df.columns]
    tf_cols = [c for c in (tf_df["idx"] + offset) if c in align_df.columns]

    ts_align = align_df.loc[lambda_max, ts_cols]
    tf_align = align_df.loc[lambda_max, tf_cols]

    delta = (ts_align.mean() - tf_align.mean()) * 100
    _, p_val = proportions_ztest(
        [ts_align.sum(), tf_align.sum()],
        [len(ts_align), len(tf_align)],
        alternative="larger"
    )

    return {
        "n_TS": len(ts_align),
        "n_TF": len(tf_align),
        "TS_align_%": round(ts_align.mean() * 100, 2),
        "TF_align_%": round(tf_align.mean() * 100, 2),
        "Delta_TS_TF_%": round(delta, 2),
        "p_value": round(p_val, 4),
    }

# Map alignment file language names -> accuracy file ISO codes
lang_map = {"italian": "it", "thai": "th"}
metric_cols = {
    "dali": "dali_max_lamda",
    "dali_strong": "dalist_max_lamda",
    # Uncomment once you add mexa_max_lamda to xcopa_max_lamda_df:
    # "mexa": "mexa_max_lamda",
}

results = []
for _, row in xcopa_max_lamda_df.iterrows():
    align_lang = row["target_languages"]  # 'italian' or 'thai'
    eval_lang = lang_map[align_lang]       # 'it' or 'th'
    ts_df, tf_df = separate_ts_tf(eval_lang, num_samples=167)

    for metric, col_name in metric_cols.items():
        lambda_max = int(row[col_name])
        # Finds the matching json file (e.g., *italian*.json or *thai*.json)
        json_files = list(Path(f"../alignment_results/Llama3.2-1B/xcopa/{metric}").glob(f"*{align_lang}*.json"))
        if not json_files:
            print(f"Missing file for {metric} - {align_lang}")
            continue

        stats = compute_delta_from_dfs(json_files[0], ts_df, tf_df, lambda_max)
        results.append({
            "language": align_lang,
            "metric": metric,
            "lambda_max": lambda_max,
            **stats
        })

xcopa_delta_df = pd.DataFrame(results)
xcopa_delta_df

,language,metric,lambda_max,n_TS,n_TF,TS_align_%,TF_align_%,Delta_TS_TF_%,p_value
0,italian,dali,5,51,44,82.35,59.09,23.26,0.0061
1,italian,dali_strong,0,51,44,1.96,2.27,-0.31,0.5421
2,thai,dali,5,55,17,38.18,23.53,14.65,0.1337
3,thai,dali_strong,5,55,17,12.73,0.00,12.73,0.0608


In [46]:
lang_df = pd.read_json("../results/accuracy/Llama3.2-1B/belebele/belebele_eval_Llama3.2-1B_deu_Latn_ABCD.jsonl", lines=True)

In [47]:
lang_df

,idx,acc,selected_logits,correct_ans
0,0,0,"[10.84375, 10.2578125, 11.75, 10.2578125]",0
1,1,0,"[10.6328125, 10.1875, 11.9296875, 10.1953125]",0
2,2,0,"[11.125, 10.734375, 12.3359375, 10.46875]",1
3,3,0,"[10.78125, 10.171875, 12.0546875, 9.9921875]",1
4,4,1,"[11.3203125, 10.796875, 12.4453125, 10.25]",2
...,...,...,...,...
895,895,0,"[11.3671875, 10.453125, 11.953125, 9.921875]",0
896,896,0,"[10.65625, 10.109375, 11.4765625, 9.875]",1
897,897,0,"[10.859375, 10.09375, 11.5078125, 9.6015625]",1
898,898,0,"[11.40625, 10.671875, 12.5625, 11.015625]",1


In [ ]:
select_logit = lang_df["selected_logits"]
select_choice = []
for logit in select_logit:
    select_choice.append(logit.index(max(logit)))
    # print(logit.index(max(logit)))

lang_df["select_choice"] = select_choice

2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
3
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
0
2
2
2
2
2
2
2
2
0
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
0
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
3
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
2
3
2
2
2
2
2
2
2
2
2
2
2
2
2
